# Python Stored Procedure and UDF
---
This notebook demonstrates Snowpark Python, creating a stored procedure that trains and saves a machine learning model, and a Python UDF that uses the trained model to make predictions. Both the model training and the UDF operate within the Python execution engine inside a Snowflake virtual warehouse.

<strong>NOTE:</strong>
We use sklearn logistic regression in this demo. It is likely that further exploration, other models, and model tuning will improve prediction accuracy.

<strong>NOTE 2:</strong> Please shut down all running kernels in your JupyterLab instance before proceeding with this lab. This Snowpark Python walkthrough is intensive in its use of memory resources in your environment. Without this allocation you may encounter out of memory exceptions.

### Steps below:

1. Connect to Snowflake
2. Prepare to save a trained model in a Snowflake stage
3. Create a function that trains, evaluates, and saves a model
4. Create and register the function in a stored procedure
5. Call the procedure
6. Define and save a Python UDF that loads and invokes the trained model
7. Make predictions with the Python UDF

---
#### 1. Connect to Snowflake

In [1]:
import snowflake.snowpark
from snowflake.snowpark.functions import * 
from snowflake.snowpark.session import Session
from snowflake.snowpark.types import *

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

import pandas as pd
import sys

In [2]:
# My code for Snowflake account connection

CONFIG_DIR = '/Users/richardkirk/.ssh'
CONFIGFILE = CONFIG_DIR + '/sf_config'


# Load configuration file
with open(CONFIGFILE) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

Confirm connection and the session's Snowflake context.

In [3]:
print( session.get_current_role() , session.get_current_warehouse() ,session.get_current_database() ,session.get_current_schema())

"OPENFLOW_ADMIN" "ML_MODEL_WH" None None


#### 2. Prepare to save a trained model in a Snowflake stage

In [4]:
#My extra bits:
session.sql('use database RKIRK_DB').collect()
session.sql('use role ACCOUNTADMIN').collect()


# Create internal stages for models and python code
session.sql('''
    create or replace stage model_data
''').collect()

session.sql('''
    create or replace stage python_load
''').collect()

[Row(status='Stage area PYTHON_LOAD successfully created.')]

In [5]:
# Function to save model to a Snowflake stage
import io
import joblib
def save_file(session, model, path):
  input_stream = io.BytesIO()
  joblib.dump(model, input_stream)
  session._conn._cursor.upload_stream(input_stream, path)
  return "successfully created file: " + path

#### 3. Create a function that trains, evaluates, and saves a model

In [6]:
# Declare a function that will be saved as a stored procedure in Snowflake
def train_customer_churn_model(session: snowflake.snowpark.Session) -> str:
    
    # Get customer churn data
    churn_df = session.table('DATA_SCIENCE_DB.PUBLIC.CUSTOMER_CHURN') \
        .withColumn('CHURNED', col('CHURNED').cast(FloatType())) \
        .withColumn('TENURE', col('TENURE').cast(FloatType())) \
        .withColumn('NUM_OF_PRODUCTS', col('NUM_OF_PRODUCTS').cast(FloatType())) \
        .withColumn('HAS_AIRLINE_CREDIT_CARD', col('HAS_AIRLINE_CREDIT_CARD').cast(FloatType())) \
        .withColumn('IS_ACTIVE_MEMBER', col('IS_ACTIVE_MEMBER').cast(FloatType())) \
        .withColumn('ESTIMATED_SALARY', col('ESTIMATED_SALARY').cast(FloatType())) \
        .select('CHURNED', 'CREDIT_SCORE', 'GEOGRAPHY', 'GENDER', 'AGE', 'TENURE',
                'MILEAGE_POINTS', 'NUM_OF_PRODUCTS', 'HAS_AIRLINE_CREDIT_CARD',
                'IS_ACTIVE_MEMBER', 'ESTIMATED_SALARY')

    # split data 
    train, test = churn_df.random_split([0.8, 0.2], seed=42)
    
    # Create Pandas DataFrames for training and evaluation
    train_pd_x = train.drop('CHURNED').toPandas()
    train_pd_y = train.select('CHURNED').toPandas()
    test_pd_x = test.drop('CHURNED').toPandas()
    test_pd_y = test.select('CHURNED').toPandas()
    
    # Build machine learning pipeline in sklearn
    numeric_columns = ['CREDIT_SCORE', 'AGE', 'TENURE', 'MILEAGE_POINTS', 'NUM_OF_PRODUCTS', 
                  'HAS_AIRLINE_CREDIT_CARD', 'ESTIMATED_SALARY']
    categorical_columns = ['GEOGRAPHY', 'GENDER']
    
    preprocessor = ColumnTransformer([
        ("num", MinMaxScaler(), numeric_columns),
        ("cat", OneHotEncoder(), categorical_columns)
    ])
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(class_weight = 'balanced'))
    ])
    
    # Fit the model on the train sample
    pipeline.fit(train_pd_x, train_pd_y.values.ravel())
    
    # Score the test data and measure accuracy
    predicted = pipeline.predict(test_pd_x)
    est_accuracy = str(metrics.accuracy_score(test_pd_y, predicted))
    
    # Fit the model to all data before saving
    churn_pd_x = churn_df.drop('CHURNED').toPandas()
    churn_pd_y = churn_df.select('CHURNED').toPandas()
    pipeline.fit(churn_pd_x, churn_pd_y.values.ravel())
    
    # Save model to Snowflake stage
    save_file(session, pipeline, "@MODEL_DATA/churn_classifier.joblib")
    
    # Return estimated model accuracy
    return_string = 'Model saved. Estimated accuracy: ' + est_accuracy
    return return_string

#### 4. Create and register the function in a stored procedure
See [Creating and Registering a Named Stored Procedure](https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-sprocs.html#creating-and-registering-a-named-stored-procedure)

In [ ]:
# MyNote: this took approx 1.5 minutes
# Creates proc RKIRK_DB.PUBLIC.TRAIN_CUSTOMER_CHURN_MODEL

session.clear_packages()
session.add_packages('snowflake-snowpark-python', 'scikit-learn', 'pandas', 'numpy')

# Register the stored procedure
session.sproc.register(name='train_customer_churn_model',
                       func=train_customer_churn_model, 
                       replace=True, 
                       is_permanent=True, 
                       stage_location='python_load')

The version of package 'numpy' in the local environment is 2.4.6, which does not fit the criteria for the requirement 'numpy'. Your UDF might not work when the package version is different between the server and your local environment.


#### 5. Call the procedure
Note this can be invoked in any Snowflake session, to retrain the model with the content of the DATA_SCIENCE_DB.PUBLIC.CUSTOMER_CHURN table. A standalone invocation would be:

    call train_customer_churn_model();

In [8]:
session.call('train_customer_churn_model')

'Model saved. Estimated accuracy: 0.6242632612966601'

#### 6. Define and save a Python UDF that loads and invokes the trained model

In [ ]:
# MyNote: This function uses pickled model that is in snowflake stage

# Define function
# Note, the number and data types of the function arguments
#  match the columns in the DATA_SCIENCE_DB.NEW_DATA.CUSTOMERS
#  table.
def predict_churn(CUSTOMER_ID: str,           # Not used
                  SURNAME_MASKED: str,        # Not used
                  CREDIT_SCORE: int,
                  GEOGRAPHY: str,
                  GENDER: str,
                  AGE: int,
                  TENURE: str,
                  MILEAGE_POINTS: int,
                  NUM_OF_PRODUCTS: str,
                  HAS_AIRLINE_CREDIT_CARD: str,
                  IS_ACTIVE_MEMBER: str,
                  ESTIMATED_SALARY: str) -> bool:
    
    df_row = pd.DataFrame([locals()])
    
    model_file = sys._xoptions.get("snowflake_import_directory") + 'churn_classifier.joblib'

    with open(model_file,'rb') as f:
        model = joblib.load(f)
        return model.predict(df_row)

# Save as a Python UDF
session.clear_imports()
session.add_import('@MODEL_DATA/churn_classifier.joblib')
session.udf.register(name='predict_churn',
                     func=predict_churn,
                     is_permanent=True, 
                     stage_location='python_load',
                     replace = True)

#### 7. Make predictions with the Python UDF

In [10]:
# Predict churn for 10 customers in the DATA_SCIENCE_DB.NEW_DATA.CUSTOMERS table
session.table('DATA_SCIENCE_DB.NEW_DATA.CUSTOMERS') \
    .sort(col('CUSTOMER_ID')) \
    .limit(7) \
    .select( \
        'CUSTOMER_ID',
        'SURNAME_MASKED',
        call_udf (
            'PREDICT_CHURN',
            col('CUSTOMER_ID'),
            col('SURNAME_MASKED'),
            col('CREDIT_SCORE'),
            col('GEOGRAPHY'),
            col('GENDER'),
            col('AGE'),
            col('TENURE'),
            col('MILEAGE_POINTS'),
            col('NUM_OF_PRODUCTS'),
            col('HAS_AIRLINE_CREDIT_CARD'),
            col('IS_ACTIVE_MEMBER'),
            col('ESTIMATED_SALARY')).alias('PREDICT_CHURN?')) \
    .show()

-------------------------------------------------------
|"CUSTOMER_ID"  |"SURNAME_MASKED"  |"PREDICT_CHURN?"  |
-------------------------------------------------------
|1              |Surname_000001    |False             |
|2              |Surname_000002    |False             |
|3              |Surname_000003    |True              |
|4              |Surname_000004    |True              |
|5              |Surname_000005    |False             |
|6              |Surname_000006    |False             |
|7              |Surname_159977    |False             |
-------------------------------------------------------



### MyNote: SQL equivlant
This can be run directly in snowsight:
```sql
SELECT
    CUSTOMER_ID,
    SURNAME_MASKED,
    PREDICT_CHURN(
        CUSTOMER_ID,
        SURNAME_MASKED,
        CREDIT_SCORE,
        GEOGRAPHY,
        GENDER,
        AGE,
        TENURE,
        MILEAGE_POINTS,
        NUM_OF_PRODUCTS,
        HAS_AIRLINE_CREDIT_CARD,
        IS_ACTIVE_MEMBER,
        ESTIMATED_SALARY
    ) AS "PREDICT_CHURN?"
FROM DATA_SCIENCE_DB.NEW_DATA.CUSTOMERS
ORDER BY CUSTOMER_ID
LIMIT 7;
```


The <code>predict_churn()</code> function can be called in any standard SQL statement, as shown here:

In [ ]:
session.sql('''
    select customer_id, 
           surname_masked,
           predict_churn(
                CUSTOMER_ID,
                SURNAME_MASKED,
                CREDIT_SCORE,
                GEOGRAPHY,
                GENDER,
                AGE,
                TENURE,
                MILEAGE_POINTS,
                NUM_OF_PRODUCTS,
                HAS_AIRLINE_CREDIT_CARD,
                IS_ACTIVE_MEMBER,
                ESTIMATED_SALARY) as "PREDICT_CHURN?"
    from data_science_db.new_data.customers
    order by customer_id
    limit 7 offset 7
''').show()

The signature of the <code>predict_churn()</code> function exactly matches the columns in the DATA_SCIENCE_DB.NEW_DATA.CUSTOMERS table, so the argument list to the function can be simplified:

In [11]:
session.sql('''
    select customer_id, 
           surname_masked,
           predict_churn(*) as "PREDICT_CHURN?"
    from data_science_db.new_data.customers
    order by customer_id
    limit 7 offset 14
''').show()

-------------------------------------------------------
|"CUSTOMER_ID"  |"SURNAME_MASKED"  |"PREDICT_CHURN?"  |
-------------------------------------------------------
|15             |Surname_702264    |False             |
|16             |Surname_434369    |False             |
|17             |Surname_766829    |False             |
|18             |Surname_707842    |False             |
|19             |Surname_358169    |False             |
|20             |Surname_946017    |False             |
|21             |Surname_999415    |False             |
-------------------------------------------------------

